# SDialog dependencies

In [ ]:
# Setup the environment depending on weather we are running in Google Colab or Jupyter Notebook
import os
from IPython import get_ipython

if "google.colab" in str(get_ipython()):
    print("Running on CoLab")

    # Installing sdialog
    !git clone https://github.com/qanastek/sdialog.git
    %cd sdialog
    %pip install -e .
    %cd ..
else:
    print("Running in Jupyter Notebook")
    # Little hack to avoid the "OSError: Background processes not supported." error in Jupyter notebooks"
    get_ipython().system = os.system

## Local installation

Create a `.venv` using the root `requirement.txt` file and Python `3.11.14`

In [ ]:
from sdialog import Dialog
from IPython.display import display

# Load an existing dialogue

In order to run the next steps in a fast manner, we will start from an existing dialog generated using previous tutorials:

In [ ]:
path_dialog = "../../tests/data/demo_dialog_doctor_patient.json"

if not os.path.exists(path_dialog) and not os.path.exists("./demo_dialog_doctor_patient.json"):
    !wget https://raw.githubusercontent.com/qanastek/sdialog/refs/heads/main/tests/data/demo_dialog_doctor_patient.json
    path_dialog = "./demo_dialog_doctor_patient.json"

original_dialog = Dialog.from_file(path_dialog)
original_dialog.print()

# Tutorial 18: Task annotator

The key objective of this tutorial is to apply different microphone impulse responses to the audio obtains after the accoustics simulation of the room, allowing you to hear how the dialogue would sound as if recorded on various real-world devices.

In [ ]:
from sdialog.audio.dialog import AudioDialog

In [ ]:
test_dialog = AudioDialog.from_dialog(original_dialog)
print(test_dialog.get_annotations())

Convert the original dialog into a audio enhanced dialog

In [ ]:
!pip install -q kokoro>=0.9.4
!apt-get -qq -y install espeak-ng > /dev/null 2>&1

In [ ]:
from sdialog.audio.tts import KokoroTTS
tts_engine = KokoroTTS()

In [ ]:
new_dialog: AudioDialog = original_dialog.to_audio(
    perform_room_acoustics=True,
    tts_engine=tts_engine,
    dialog_dir_name="counting_turns"
)

You can create your own task annotator:

In [ ]:
import logging
from typing import Any

from sdialog import Dialog
from sdialog.annotators import Annotator, TaskModality

class TurnCountingAnnotator(Annotator):
    """
    Annotator for counting the number of turns in a dialog.
    """

    def get_modality(self) -> list[TaskModality]:
        return [
            TaskModality.TEXT_TO_TEXT
        ]

    def get_task_name(self) -> str:
        return "turn_counting"

    def get_requirements(self) -> list[Annotator]:
        return []

    def save(self, data: Any, args: dict[str, Any] = {}) -> None:
        import pandas as pd

        if args is None or "save_path" not in args or args["save_path"] is None:
            logging.warning("[TurnCountingAnnotator] No 'save_path' provided, skipping saving")
            return

        df = pd.DataFrame(data)
        df = df[["question", "answer"]]
        df.to_csv(args["save_path"], index=False)

        logging.info(
            "[TurnCountingAnnotator] "
            f"Data saved to {args['save_path']}"
        )

    def annotate(self, dialog: Dialog, args: dict[str, Any] = {}) -> Dialog:

        logging.info("[TurnCountingAnnotator] Annotating dialog for turn counting tasks")

        dialog = self.check_requirements(dialog)

        _annotations = {
            "data": [
                {
                    "question": "How many turns are there in this dialog?",
                    "answer": str(len(dialog.turns))
                }
            ],
            "modality": self._modality
        }

        dialog.add_annotations(self._task_name, _annotations)

        logging.info("[TurnCountingAnnotator] Annotations done!")

        self.save(data=_annotations["data"], args=args)

        return dialog

In [ ]:
from sdialog.annotators import dialog2annotations

In [ ]:
new_dialog = dialog2annotations(
    new_dialog,
    [
        (TurnCountingAnnotator(), {"save_path": "./outputs_annotations/turn_counting.csv"})
    ]
)
print(new_dialog.get_annotations())